
### Ziel dieser Datei
Berechnungen auf Basis der Grundlagedaten. Hier sollen alle verschnitte/buffer usw. berechnet werden, um überhaupt die Gebiete welche in Frage kommen zu definieren. Am schluss also 1 datei mit allen Flächen die überhaupt in Frage kommen.

### Datenquellen
Die 4 exportierten GeoJSON-Dateien aus Datei 1 einlesen. Es sollte hier keine externen Dateien mehr brauchen.

### Grober Codeaufbau
1. Ausschlusszonen: Bestimmte Neigung ab gewissem Grad oder wenn nicht möglich zumindest die Gefahrenzonen (Klippen) und die Schutzgebiete mit einem Buffer ringsherum komplett ausschliessen. Damit für diese Gebiete gar nicht erst die Möglichkeit besteht, dort ein geeigneter Standort zu zeigen.
2. Ausschlusszonen auschneiden: Die in 1. definierten Zonen sollen mit den infragekommenden Flächen (Wälder/Wiesen) verschnitten werden.
3. Abfrage nach der Infrastruktur: z.B. Puffer um Haltestellen und Bauernhöfe erstellen. Wenn möglich soll dieser Puffer später auch selbst angepasst werden (je nachdem wie wichtig einem dies ist/wie nahe man sein möchte).
4. Alle Lagerflächen, damit sie diesen Kriterien 1-3 entsprechen rausfiltern.

### Export & Übernahme für die Nächste Datei 3
* Finale Liste aller geeigneten Flächen in der Schweiz im GeoJSON-Format mit sinnvoller Bezeichnung.

In [1]:
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# ============================================================
# 1. Einstellungen
# ============================================================

# Ordner
DATA_DIR = Path("data")
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

# Ziel-Koordinatensystem: LV95 Schweiz
TARGET_CRS = "EPSG:2056"

# ------------------------------------------------------------
# Dateinamen aus Notebook 1 anpassen
# ------------------------------------------------------------

PATH_WALD_WIESE = DATA_DIR / "wald_wiese.geojson"
PATH_GEROELL_FELSEN = DATA_DIR / "geroell_felsen.geojson"
PATH_NATURSCHUTZ = DATA_DIR / "naturschutzgebiete.geojson"

PATH_BAUERNHOF = DATA_DIR / "bauernhoefe.geojson"
PATH_HYDRANT = DATA_DIR / "hydranten.geojson"
PATH_OEV = DATA_DIR / "oev_haltestellen.geojson"

# ------------------------------------------------------------
# Analyse-Parameter
# ------------------------------------------------------------

# Puffer um Ausschlusszonen
BUFFER_NATURSCHUTZ_M = 50
BUFFER_GEROELL_FELSEN_M = 0

# Kleinste Fläche, die als Lagerfläche sinnvoll ist
# Falls ihr nichts entfernen wollt: auf 0 setzen
MIN_FLAECHE_M2 = 5000

# Filter-Distanzen
MAX_DIST_BAUERNHOF_M = 1500
MAX_DIST_HYDRANT_M = 500
MAX_DIST_OEV_M = 1500

# False = alle Must-have-Flächen exportieren, Filter nur als Attribute speichern
# True = nur Flächen exportieren, welche alle Filter erfüllen
NUR_FLAECHEN_MIT_ALLEN_FILTERKRITERIEN = False

In [3]:
# ============================================================
# 2. Hilfsfunktionen
# ============================================================

def read_layer(path, target_crs=TARGET_CRS):
    """
    GeoJSON einlesen, CRS prüfen, nach LV95 transformieren
    und ungültige/leere Geometrien entfernen.
    """
    if not path.exists():
        raise FileNotFoundError(f"Datei nicht gefunden: {path}")

    gdf = gpd.read_file(path)

    if gdf.empty:
        print(f"Achtung: {path.name} ist leer.")
        return gdf

    if gdf.crs is None:
        print(f"Achtung: {path.name} hat kein CRS. Es wird {target_crs} angenommen.")
        gdf = gdf.set_crs(target_crs)
    else:
        gdf = gdf.to_crs(target_crs)

    gdf = gdf[gdf.geometry.notna()]
    gdf = gdf[~gdf.geometry.is_empty]

    # Geometrien reparieren
    gdf["geometry"] = gdf.geometry.buffer(0)

    return gdf


def keep_polygons(gdf):
    """
    Nur Polygon- und MultiPolygon-Geometrien behalten.
    """
    return gdf[gdf.geometry.geom_type.isin(["Polygon", "MultiPolygon"])].copy()


def clean_geodataframe(gdf):
    """
    Leere Geometrien entfernen, Geometrien reparieren und explodieren.
    """
    if gdf.empty:
        return gdf

    gdf = gdf[gdf.geometry.notna()]
    gdf = gdf[~gdf.geometry.is_empty]
    gdf["geometry"] = gdf.geometry.buffer(0)
    gdf = gdf.explode(index_parts=False).reset_index(drop=True)

    return gdf


def make_buffer(gdf, buffer_m, name):
    """
    Buffer um Geometrien erstellen und zu einer Ausschlusszone zusammenfassen.
    """
    if gdf.empty:
        return gpd.GeoDataFrame(columns=["typ", "geometry"], geometry="geometry", crs=TARGET_CRS)

    buffered = gdf.copy()
    buffered["geometry"] = buffered.geometry.buffer(buffer_m)
    buffered["typ"] = name
    buffered = buffered[["typ", "geometry"]]
    buffered = clean_geodataframe(buffered)

    return buffered


def add_nearest_distance(base_gdf, target_gdf, distance_col):
    """
    Distanz jeder Fläche zum nächsten Objekt berechnen.
    Beispiel: Distanz zur nächsten ÖV-Haltestelle.
    """
    base_gdf = base_gdf.copy()

    if target_gdf.empty:
        base_gdf[distance_col] = np.nan
        return base_gdf

    base_tmp = base_gdf.copy()
    base_tmp["_orig_idx"] = base_tmp.index

    target_tmp = target_gdf[["geometry"]].copy()

    joined = gpd.sjoin_nearest(
        base_tmp,
        target_tmp,
        how="left",
        distance_col=distance_col
    )

    distances = joined.groupby("_orig_idx")[distance_col].min()
    base_gdf[distance_col] = base_gdf.index.map(distances)

    return base_gdf

In [4]:
# ============================================================
# 3. Daten einlesen
# ============================================================

wald_wiese = read_layer(PATH_WALD_WIESE)
geroell_felsen = read_layer(PATH_GEROELL_FELSEN)
naturschutz = read_layer(PATH_NATURSCHUTZ)

bauernhof = read_layer(PATH_BAUERNHOF)
hydrant = read_layer(PATH_HYDRANT)
oev = read_layer(PATH_OEV)

# Nur Polygonflächen für die eigentlichen Lagerflächen und Ausschlussflächen
wald_wiese = keep_polygons(wald_wiese)
geroell_felsen = keep_polygons(geroell_felsen)
naturschutz = keep_polygons(naturschutz)

print("Wald/Wiese:", len(wald_wiese))
print("Geröll/Felsen:", len(geroell_felsen))
print("Naturschutzgebiete:", len(naturschutz))
print("Bauernhöfe:", len(bauernhof))
print("Hydranten:", len(hydrant))
print("ÖV-Haltestellen:", len(oev))

FileNotFoundError: Datei nicht gefunden: data\wald_wiese.geojson

In [ ]:
# ============================================================
# 4. Ausschlusszonen bilden
# ============================================================

naturschutz_buffer = make_buffer(
    naturschutz,
    BUFFER_NATURSCHUTZ_M,
    "Naturschutzgebiet"
)

geroell_felsen_buffer = make_buffer(
    geroell_felsen,
    BUFFER_GEROELL_FELSEN_M,
    "Geröll/Felsen"
)

ausschlusszonen = pd.concat(
    [naturschutz_buffer, geroell_felsen_buffer],
    ignore_index=True
)

ausschlusszonen = gpd.GeoDataFrame(
    ausschlusszonen,
    geometry="geometry",
    crs=TARGET_CRS
)

ausschlusszonen = clean_geodataframe(ausschlusszonen)

print("Ausschlusszonen vor dissolve:", len(ausschlusszonen))

# Alle Ausschlussflächen zu einer Fläche zusammenfassen
if not ausschlusszonen.empty:
    ausschlusszonen_dissolved = ausschlusszonen.dissolve()
else:
    ausschlusszonen_dissolved = ausschlusszonen.copy()

print("Ausschlusszonen nach dissolve:", len(ausschlusszonen_dissolved))

In [ ]:
# ============================================================
# 5. Ausschlusszonen aus Wald/Wiese ausschneiden
# ============================================================

if not ausschlusszonen_dissolved.empty:
    lagerflaechen = gpd.overlay(
        wald_wiese,
        ausschlusszonen_dissolved[["geometry"]],
        how="difference",
        keep_geom_type=True
    )
else:
    lagerflaechen = wald_wiese.copy()

lagerflaechen = clean_geodataframe(lagerflaechen)

print("Flächen nach Ausschluss:", len(lagerflaechen))

In [ ]:
# ============================================================
# 6. Fläche berechnen und kleine Restflächen entfernen
# ============================================================

lagerflaechen["flaeche_m2"] = lagerflaechen.geometry.area
lagerflaechen["flaeche_ha"] = lagerflaechen["flaeche_m2"] / 10000

lagerflaechen = lagerflaechen[
    lagerflaechen["flaeche_m2"] >= MIN_FLAECHE_M2
].copy()

lagerflaechen = lagerflaechen.reset_index(drop=True)

print("Flächen nach Mindestfläche:", len(lagerflaechen))
print("Gesamtfläche in ha:", round(lagerflaechen["flaeche_ha"].sum(), 2))

In [ ]:
# ============================================================
# 7. Distanz zu Filter-Kriterien berechnen
# ============================================================

lagerflaechen = add_nearest_distance(
    lagerflaechen,
    bauernhof,
    "dist_bauernhof_m"
)

lagerflaechen = add_nearest_distance(
    lagerflaechen,
    hydrant,
    "dist_hydrant_m"
)

lagerflaechen = add_nearest_distance(
    lagerflaechen,
    oev,
    "dist_oev_m"
)

lagerflaechen[[
    "dist_bauernhof_m",
    "dist_hydrant_m",
    "dist_oev_m"
]].head()

In [ ]:
# ============================================================
# 8. Filter-Kriterien als Attribute speichern
# ============================================================

lagerflaechen["nahe_bauernhof"] = lagerflaechen["dist_bauernhof_m"] <= MAX_DIST_BAUERNHOF_M
lagerflaechen["nahe_hydrant"] = lagerflaechen["dist_hydrant_m"] <= MAX_DIST_HYDRANT_M
lagerflaechen["nahe_oev"] = lagerflaechen["dist_oev_m"] <= MAX_DIST_OEV_M

# einfacher Score: 0 bis 3
lagerflaechen["filter_score"] = (
    lagerflaechen["nahe_bauernhof"].astype(int)
    + lagerflaechen["nahe_hydrant"].astype(int)
    + lagerflaechen["nahe_oev"].astype(int)
)

lagerflaechen[[
    "flaeche_ha",
    "dist_bauernhof_m",
    "dist_hydrant_m",
    "dist_oev_m",
    "nahe_bauernhof",
    "nahe_hydrant",
    "nahe_oev",
    "filter_score"
]].head()

In [ ]:
# ============================================================
# 9. Optional: nur Flächen behalten, die alle Filter erfüllen
# ============================================================

if NUR_FLAECHEN_MIT_ALLEN_FILTERKRITERIEN:
    lagerflaechen_final = lagerflaechen[
        (lagerflaechen["nahe_bauernhof"])
        & (lagerflaechen["nahe_hydrant"])
        & (lagerflaechen["nahe_oev"])
    ].copy()
else:
    lagerflaechen_final = lagerflaechen.copy()

lagerflaechen_final = lagerflaechen_final.reset_index(drop=True)

print("Finale Lagerflächen:", len(lagerflaechen_final))

In [ ]:
# ============================================================
# 10. ID und sinnvolle Klassierung ergänzen
# ============================================================

lagerflaechen_final["lagerplatz_id"] = range(1, len(lagerflaechen_final) + 1)

def bewertung(score):
    if score == 3:
        return "sehr geeignet"
    elif score == 2:
        return "geeignet"
    elif score == 1:
        return "bedingt geeignet"
    else:
        return "weniger geeignet"

lagerflaechen_final["bewertung"] = lagerflaechen_final["filter_score"].apply(bewertung)

lagerflaechen_final[[
    "lagerplatz_id",
    "flaeche_ha",
    "filter_score",
    "bewertung"
]].head()

In [ ]:
# ============================================================
# 11. Export für Notebook 3
# ============================================================

output_path = OUT_DIR / "geeignete_lagerflaechen.geojson"

lagerflaechen_final.to_file(
    output_path,
    driver="GeoJSON"
)

print(f"Export abgeschlossen: {output_path}")